In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)


In [2]:
files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [ ]:
## question 1
len(documents)

72

In [15]:
documents[0].keys()

dict_keys(['content', 'filename'])

In [5]:
from src.searching_tool import fit_docs

In [6]:
index = fit_docs(
    documents,
    text_fields=["content"],
    keyword_fields=["filename"]
)

In [ ]:
## question 2
index.search(
    "How does the agentic loop keep calling the model until it stops?"
)[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [10]:
%load_ext autoreload
%autoreload 2

In [15]:
from src.rag_helper import RAGHomework

In [14]:
from langchain_ollama import ChatOllama

In [17]:
assistant = RAGHomework(
    index=index,
    llm_client=ChatOllama,
    model="qwen3.5:0.8b",
)

In [21]:
response = assistant.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

In [28]:
# question 3 (see tokens and identify the closets)
response.usage_metadata

{'input_tokens': 4096, 'output_tokens': 1705, 'total_tokens': 5801}

In [24]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [27]:
## question 4
len(chunks)

295

In [32]:
index_chunks = fit_docs(
    chunks,
    text_fields=["content"],
    keyword_fields=["filename"]
)

assistant_chunks = RAGHomework(
    index=index_chunks,
    llm_client=ChatOllama,
    model="qwen3.5:0.8b",
)

In [34]:
response_chunks = assistant_chunks.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

In [35]:
# question 6 (see tokens and identify the closets)
response_chunks.usage_metadata

{'input_tokens': 2393, 'output_tokens': 1393, 'total_tokens': 3786}

In [37]:
print(response_chunks.content)

Based on the provided context, the agentic loop keeps calling the model until it stops through the following mechanisms:

**The Core Loop Mechanism**
The agentic loop is a **while True loop** that continuously calls the model. The loop uses a flag variable (`has_function_calls`) to track whether the model has already performed tool calls in the current iteration.

1. The agent performs a tool call via `openai_client.responses.create()`
2. If the tool call results in a function call (`function_call` type), it calls `make_call()` and stores it in the message list
3. The flag `has_function_calls` is updated to **True** after calling the tool
4. After the loop finishes, it increments the iteration counter and checks `has_function_calls` again
5. If the flag is **False** in the current iteration, the loop **breaks**, signaling the agent has reached a final answer without more tool calls

**Key Logic**
- The loop keeps calling the model until it returns a response without any function calls.

In [38]:
### agentic workflow 
import json

from typing import List, Dict

from langchain_core.tools import tool

from langchain_ollama import ChatOllama

from langchain_core.messages import (
    HumanMessage,
    ToolMessage,
    SystemMessage
)

In [39]:
@tool
def search(query: str) -> List[Dict]:
    """Search the FAQ database for entries matching the given query.
    Args:
        query: Search query text to look up in the course FAQ.
    Returns:
        List[Dict]: List of documents that are relevant to the query
    """
    boost_dict = {"question": 3.0, "section": 0.5}

    results = index_chunks.search(
        query, num_results=5, boost_dict=boost_dict
    )

    return results

tools = [search]
llm_with_tools = ChatOllama(
    model="qwen3.5:0.8b",
    reasoning=True,
).bind_tools(tools, tool_choice=True)

In [41]:
## agentic workflow

def make_call(call):
    args = call["args"]
    if call["name"] == "search":
        result = search.invoke(args)
    result_json = json.dumps(result, indent=2)
    return ToolMessage(
        content=result_json, 
        tool_call_id=call["id"]
    )

def agent_loop(instructions, question, llm_w_tools) -> str:
    messages = [
        SystemMessage(content=instructions),
        HumanMessage(content=question)
    ]

    it = 1
    tc = 0
    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = llm_w_tools.invoke(messages)
        messages.append(response)

        if response.tool_calls:
            tc += 1
            print(f"Tool called #{it}...")

            has_function_calls = True
            
            tool_calls = response.tool_calls[0]
            print('tool_calls:', tool_calls["name"], tool_calls["args"])
            call_output = make_call(tool_calls)
            messages.append(call_output)

        elif response.content:
            print('ASSISTANT:')
            last_answer = response.content
            print(response.content)

        it = it + 1
        
        if has_function_calls == False: 
            break

    return last_answer

In [42]:
agent_instructions = """You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."""
question_agent= """How does the agentic loop work, and how is it different from plain RAG?"""

agent_response = agent_loop(
    agent_instructions,
    question_agent,
    llm_with_tools
)

iteration #1...
Tool called #1...
tool_calls: search {'query': 'agentic loop AI'}
iteration #2...
Tool called #2...
tool_calls: search {'query': 'agentic loop vs RAG comparison'}
iteration #3...
Tool called #3...
tool_calls: search {'query': 'agentic loop RAG function calling'}
iteration #4...
Tool called #4...
tool_calls: search {'query': 'agentic loop FAQ'}
iteration #5...
Tool called #5...
tool_calls: search {'query': 'while loop'}
iteration #6...
Tool called #6...
tool_calls: search {'query': 'while True loop'}
iteration #7...
Tool called #7...
tool_calls: search {'query': 'agent loop while True'}
iteration #8...
Tool called #8...
tool_calls: search {'query': 'while True loop'}
iteration #9...
ASSISTANT:
# Understanding the While True Loop in Agent-Based Systems

Based on the documentation we reviewed, here's how the **while True loop** works in an agent-based approach:

## Core Concept

This loop is the **core agent loop** that continuously searches for information from the model 